#**Analysis of Data Manipulation and Algorithm Optimization Strategies Using MIMIC-IV-ED v2.2**

In this research project, we will explore various methods and approaches for data manipulation and algorithm optimization. Our goal is to analyze different optimal scenarios under various conditions, leveraging multiple techniques to refine our understanding and improve outcomes. We will be working with the MIMIC-IV-ED v2.2 dataset, which provides rich data for our analysis.

Throughout this project, we will apply several methods to preprocess and analyze the data, including different algorithms and strategies. It is important to note that some methods, particularly Methods 2 and 3, may potentially compromise the model's ability to generalize to new data. This trade-off is a key consideration in our analysis, and we will approach each method with the understanding that the adjustments made are aimed at optimizing performance within the specific context of this study.

Our exploration is driven by the objective of identifying the most effective strategies for various scenarios, while being mindful of the potential impact on generalization capabilities.

#First method for addressing the problem

We will employ a statistical method to remove outliers and explore various algorithms, striving to enhance the final outcomes while ensuring the ability to generalize to new data is preserved.

##Data loading



In [0]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

%cd /content/drive/MyDrive/data
df1 = pd.read_csv('triage.csv')
df2=pd.read_csv('edstays.csv')
df_train = pd.merge(df1, df2, on='stay_id', how='left')

**Destription of each columns**

**subject_id**: Unique identifier for each patient.

**stay_id**: Unique identifier for each hospital stay or visit.

**temperature**: The patient's body temperature, usually measured in Fahrenheit.

**heartrate**: The patient's heart rate, measured in beats per minute (bpm).

**resprate**: The patient's respiratory rate, measured in breaths per minute.

**o2sat**: The patient's oxygen saturation level, typically expressed as a percentage.

**sbp**: Systolic blood pressure, measured in millimeters of mercury (mmHg).

**dbp**: Diastolic blood pressure, measured in millimeters of mercury (mmHg).

**pain**:The patient's reported pain level, often measured on a scale from 0 to 10.

**acuity**:The patient's acuity level, indicating the severity of their condition, often on a scale from 1 to 5.

**chiefcomplaint**: The primary reason for the patient's visit or hospital stay, described in their own words or as recorded by medical staff.

**subject_id:** A unique identifier for each patient.

**hadm_id:** A unique identifier for each hospital admission.

**stay_id:** A unique identifier for each hospital stay.

**intime:** The timestamp indicating when the patient was admitted or entered the hospital.

**outtime:** The timestamp indicating when the patient was discharged or left the hospital.

**gender:** The gender of the patient.

**race:** The race of the patient.

**arrival_transport:** The mode of transport by which the patient arrived at the hospital.

**disposition:** The final status or outcome for the patient at the time of discharge.



In [0]:
# Ispis broja jedinstvenih vrednosti u koloni 'chiefcomplaint'
num_unique_complaints = df_train['chiefcomplaint'].nunique()
print(f"Number of unique values in 'chiefcomplaint': {num_unique_complaints}")

In [0]:
df_train.info()

In [0]:
df_train.head()

Visualization of urgency levels.

In [0]:
plt.figure(figsize=(10, 6))
sns.countplot(x='acuity', data=df_train, palette='viridis')
plt.title('Number of Patients per Urgency Level')
plt.xlabel('Urgency Level')
plt.ylabel('Number of Patients')
plt.grid(True)
plt.savefig('number_of_patients_per_urgency_level.png')
plt.show()

##Distribution of numeric variables

In [0]:
sns.set_style("whitegrid")

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_train['temperature'], kde=True, bins=30, color='blue')
plt.title('Temperature distribution')
plt.xlabel('Temperature')
plt.ylabel('Value')

plt.show()

In [0]:
sns.set_style("whitegrid")

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_train['heartrate'], kde=True, bins=30, color='blue')
plt.title('Heartrate distribution')
plt.xlabel('Heartrate')
plt.ylabel('Value')

plt.show()

In [0]:
sns.set_style("whitegrid")

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_train['resprate'], kde=True, bins=30, color='blue')
plt.title('Resprate distribution')
plt.xlabel('Resprate')
plt.ylabel('Value')

plt.show()

In [0]:
sns.set_style("whitegrid")

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_train['o2sat'], kde=True, bins=30, color='blue')
plt.title('O2sat distribution')
plt.xlabel('O2sat')
plt.ylabel('Value')

plt.show()

In [0]:
sns.set_style("whitegrid")

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_train['sbp'], kde=True, bins=30, color='blue')
plt.title('Sbp distribution')
plt.xlabel('Sbp')
plt.ylabel('Value')

plt.show()

In [0]:
sns.set_style("whitegrid")

plt.figure(figsize=(20, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_train['dbp'], kde=True, bins=30, color='blue')
plt.title('Dbp distribution')
plt.xlabel('Dbp')
plt.ylabel('Value')

plt.show()

##Chi-Square Test for identifying significant columns

In [0]:
from scipy.stats import chi2_contingency

columns_to_test = df_train.columns.drop('acuity')

# List to store significant columns
significant_columns = []

# Performing Chi-square test for each column except 'acuity'
for col in columns_to_test:
    # Creating a contingency table between 'acuity' and the current column
    contingency_table = pd.crosstab(df_train['acuity'], df_train[col])

    # Checking if the contingency table is empty or has insufficient data
    if contingency_table.shape[0] < 2 or contingency_table.shape[1] < 2:
        continue

    # Calculating Chi-square statistic and p-value
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # If p-value is less than 0.05, the column is significant
    if p < 0.05:
        significant_columns.append(col)

# Printing significant columns
if significant_columns:
    print("Significant columns according to the Chi-square test:")
    for col in significant_columns:
        print(f"- {col}")
else:
    print("No significant columns according to the Chi-square test.")

##Stratified Sampling and Data Balancing by 'acuity' classes

In [0]:
df_train.acuity.value_counts(normalize = True)

In [0]:
df_train.shape

To balance the dataset by resampling certain classes while maintaining their representativeness, and to shuffle the data to ensure better generalization for modeling.

In [0]:
# Assuming df_train is your original DataFrame
# Split the data by classes
class_1 = df_train[df_train['acuity'] == 1]
class_2 = df_train[df_train['acuity'] == 2]
class_3 = df_train[df_train['acuity'] == 3]
class_4 = df_train[df_train['acuity'] == 4]
class_5 = df_train[df_train['acuity'] == 5]

# Determine the number of samples for the downsampled classes
num_samples = len(class_4)

# Function for stratified sampling
def stratified_sample(df, n, random_state=None):
    if len(df) <= n:
        return df
    return df.groupby('acuity').apply(lambda x: x.sample(n=int(n * len(x) / len(df)), random_state=random_state)).reset_index(drop=True)

# Select random samples for class 2 and class 3 while retaining representativeness
class_2_sampled = stratified_sample(class_2, num_samples, random_state=42)
class_3_sampled = stratified_sample(class_3, num_samples, random_state=42)

# Combine all classes into one DataFrame
df_train = pd.concat([class_1, class_2_sampled, class_3_sampled, class_4, class_5])

# Shuffle the order of the data for better generalization
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

In [0]:
df_train.acuity.value_counts(normalize = True)

In [0]:
df_train.shape

##Outlier Detection using the IQR method

In [0]:
# Select only numeric columns
numeric_df = df_train.select_dtypes(include=np.number)

# Create a box plot for each numeric column
plt.figure(figsize=(15, 10))
sns.boxplot(data=numeric_df)
plt.xticks(rotation=90)
plt.title("Box Plot of Numeric Columns")
plt.show()


In [0]:
# Calculate IQR for each numeric column
Q1 = numeric_df.quantile(0.25)
Q3 = numeric_df.quantile(0.75)
IQR = Q3 - Q1

# Define lower and upper bounds for outlier detection
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower Bounds:\n", lower_bound)
print("\nUpper Bounds:\n", upper_bound)


In [0]:
# Scatter plots for outlier detection
for column in numeric_df.columns:
  plt.figure(figsize=(8, 6))
  plt.scatter(numeric_df.index, numeric_df[column])
  plt.xlabel("Index")
  plt.ylabel(column)
  plt.title(f"Scatter Plot of {column}")
  plt.show()

# Histogram with outliers highlighted
for column in numeric_df.columns:
  plt.figure(figsize=(8, 6))
  plt.hist(numeric_df[column], bins=30)
  plt.xlabel(column)
  plt.ylabel("Frequency")
  plt.title(f"Histogram of {column}")

  # Highlight outliers (using IQR method for demonstration)
  outliers = numeric_df[(numeric_df[column] < lower_bound[column]) | (numeric_df[column] > upper_bound[column])][column]
  plt.scatter(outliers, np.zeros_like(outliers), color='red', marker='x', label='Outliers')

  plt.legend()
  plt.show()


In [0]:
# Remove outliers based on IQR method
df_no_outliers = numeric_df[(numeric_df >= lower_bound) & (numeric_df <= upper_bound)].dropna()

# Recreate box plot without outliers
plt.figure(figsize=(15, 10))
sns.boxplot(data=df_no_outliers)
plt.xticks(rotation=90)
plt.title("Box Plot of Numeric Columns (Outliers Removed)")
plt.show()

# Recreate scatter plots without outliers
for column in df_no_outliers.columns:
  plt.figure(figsize=(8, 6))
  plt.scatter(df_no_outliers.index, df_no_outliers[column])
  plt.xlabel("Index")
  plt.ylabel(column)
  plt.title(f"Scatter Plot of {column} (Outliers Removed)")
  plt.show()

# Recreate histograms without outliers
for column in df_no_outliers.columns:
  plt.figure(figsize=(8, 6))
  plt.hist(df_no_outliers[column], bins=30)
  plt.xlabel(column)
  plt.ylabel("Frequency")
  plt.title(f"Histogram of {column} (Outliers Removed)")
  plt.show()


In [0]:
df_no_outliers[df_no_outliers['acuity']==1]

In [0]:
df_no_outliers[df_no_outliers['acuity']==5]

In [0]:
df_train.shape

In [0]:
df_no_outliers.shape

In [0]:
indices_no_outliers = df_no_outliers.index

# Uklanjanje redova iz df_train koji nisu u indices_no_outliers
df_train = df_train.loc[indices_no_outliers]


##TF-IDF for 'chiefcomplaint' and Label Encoding for other non-numeric columns

In [0]:
import re
def limpieza(text):
    """
    Performs text cleanup by removing non-alphanumeric characters and converting them to lowercase.

    Arguments:
    - text: text to clean.

    Returns:
    - clean_text: clean text without non-alphanumeric characters and in lower case.
    """
    if pd.isna(text):
        return ''
    text_limpio = re.sub('[\W]+', ' ', text.lower())
    return text_limpio

In [0]:
df_train['chiefcomplaint'] = df_train['chiefcomplaint'].apply(limpieza)
df_train['chiefcomplaint'] = df_train['chiefcomplaint'].str.split(',').apply(lambda x: ' '.join(x))

In [0]:
import nltk
nltk.download('stopwords')

In [0]:
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

my_stopwords = stopwords.words('english')
vectorizer_texto = TfidfVectorizer(stop_words=my_stopwords)
tfidf_matrix = vectorizer_texto.fit_transform(df_train['chiefcomplaint'])

def get_most_important_words_index(tfidf_matrix, vectorizer, n=10):
    """
    Returns the indices of the n most important words in the TF-IDF matrix.
    """
    feature_names = vectorizer.vocabulary_.keys()
    feature_names = list(feature_names)
    mean_tfidf_scores = tfidf_matrix.mean(axis=0)
    word_scores = [(col, mean_tfidf_scores[0, col]) for col in range(len(feature_names))]
    sorted_word_scores = sorted(word_scores, key=lambda x: x[1], reverse=True)
    top_word_indices = [x[0] for x in sorted_word_scores[:n]]
    return top_word_indices

important_words_index = get_most_important_words_index(tfidf_matrix, vectorizer_texto, n=800)
X_texto_reduced = tfidf_matrix[:, important_words_index]

palabras = vectorizer_texto.vocabulary_.keys()
puntajes = tfidf_matrix.mean(axis=0).tolist()[0]
palabras_puntajes = list(zip(palabras, puntajes))
palabras_puntajes.sort(key=lambda x: x[1], reverse=True)

print("Top 10 most important words:")
for palabra, puntaje in palabras_puntajes[:10]:
    print("- {} ({:.4f})".format(palabra, puntaje))

Variance analysis for different numbers of features in TF-IDF vectors.

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer

variance = []

features = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1500, 2000, 2500, 3000]

# Calculate the total TF-IDF score for all data
vectorizer_full = TfidfVectorizer(max_features=10000)  # Large number of features for the full TF-IDF
X_full = vectorizer_full.fit_transform(df_train['chiefcomplaint'])
X_full_sum = X_full.sum()

for i in features:
    tfidf_temp = TfidfVectorizer(stop_words='english', max_features=i)
    X_temp = tfidf_temp.fit_transform(df_train['chiefcomplaint'])
    variance.append(X_temp.sum() / X_full_sum)
    print(i, X_temp.sum() / X_full_sum)

# Plot variance
plt.xlabel('Number of features')
plt.ylabel('Explained variance')
plt.plot(features, variance, marker='o')
plt.axvline(x=800, linestyle='--', color='r')  # Red line at 800 features
plt.hlines(y=0.90, xmin=0, xmax=1000, linestyle='--', color='r')  # Horizontal line at 0.90
plt.title('Variance Analysis for Different Numbers of Features')
plt.show()

In [0]:
from sklearn.preprocessing import LabelEncoder
# Choose the optimal number of features (e.g., 800 from variance analysis)
optimal_features = 800

# Apply TF-IDF vectorization with the optimal number of features
vectorizer_optimal = TfidfVectorizer(stop_words='english', max_features=optimal_features)
chiefcomplaint_vectors = vectorizer_optimal.fit_transform(df_train['chiefcomplaint'])

# 3.Non-numeric columns
non_numeric_df = df_train.select_dtypes(exclude=np.number)
_
# 4. Combine all processed features
df_preprocessed = df_train.drop('chiefcomplaint', axis=1)
df_preprocessed = df_preprocessed.reset_index()
df_preprocessed = pd.concat([df_preprocessed, pd.DataFrame(chiefcomplaint_vectors.toarray(), columns=vectorizer_optimal.get_feature_names_out())], axis=1)

# Check column names
print("Column names in df_preprocessed:")
print(df_preprocessed.columns)

In [0]:
df_preprocessed.shape

In [0]:
column_names = df_preprocessed.columns

column_names_df = pd.DataFrame(column_names, columns=['Column Name'])

column_names_df.to_csv('column_names.csv', index=False)

In [0]:
df_preprocessed.shape

##Manual removal of incorrect entries for 'rever' and 'chiefcomplaint', followed by TF-IDF processing

In [0]:
condition = (df_preprocessed['fever'] != 0) & (df_preprocessed['temperature'] < 38)

indexes_to_drop = df_preprocessed[condition].index.tolist()

print("Indices of rows that meet the filtering criteria:")
print(indexes_to_drop)

In [0]:
# Assuming 'chiefcomplaint_vectors' is the sparse matrix from TF-IDF
chiefcomplaint_df = pd.DataFrame(chiefcomplaint_vectors.toarray())
df_no_outliers = df_no_outliers.reset_index()
non_numeric_df = non_numeric_df.reset_index()

# Concatenate numerical df without outliers, encoded non-numerical df, and chiefcomplaint df
df_final = pd.concat([df_no_outliers, non_numeric_df.drop('chiefcomplaint', axis=1), chiefcomplaint_df], axis=1)

# Display the final dataframe
df_final


In [0]:
df_final.shape

In [0]:
df_final = df_final.drop(indexes_to_drop)

df_final = df_final.reset_index(drop=True)


##Dropping unusable and unimportant columns

In [0]:
df_final = df_final.drop(['stay_id', 'subject_id_y'], axis=1)
df_final


In [0]:
df_final = df_final.drop(['intime', 'outtime'], axis=1)
df_final


In [0]:
df_final = df_final.drop(['disposition', 'hadm_id','subject_id_x'], axis=1)
df_final

##'Pain' column

In [0]:
print(df_final['pain'].dtype)

Convert 'pain' to numerical (values that we can , and drop others).

In [0]:
# Attempt to convert 'pain' column to numeric, coercing errors
df_final['pain'] = pd.to_numeric(df_final['pain'], errors='coerce')

# Drop rows with missing values (originally non-numeric) in 'pain' column
df_final = df_final.dropna(subset=['pain'])

print(df_final['pain'].dtype)


In [0]:
unique_values = df_final['pain'].unique()
count_values = df_final['pain'].nunique()

print("Unique Values in 'pain' Column:", unique_values)
print("Number of Unique Values:", count_values)


In [0]:
# Count occurrences of each unique value
pain_counts = df_final['pain'].value_counts()

print(pain_counts)


Leave only relevant pain values.

In [0]:
# Filter the DataFrame to include only rows with pain values 0, 1, 2, 3, 4, 5, 7, 8, 9, or 10
relevant_pain_values = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10]
df_final = df_final[df_final['pain'].isin(relevant_pain_values)]

# Display the filtered DataFrame
df_final


##Encoding non-numerical columns

In [0]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pandas.api.types as ptypes

# Pretpostavljamo da je df_final vaš DataFrame

# Kreiranje LabelEncoder instance
le = LabelEncoder()

# Iteriranje kroz kolone DataFrame-a
for column in df_final.columns:
    if ptypes.is_string_dtype(df_final[column]):  # Provera da li je kolona nenumerička
        df_final[column] = le.fit_transform(df_final[column])

# Prikazivanje prvih nekoliko redova DataFrame-a za proveru
df_final.head()


##Handling missing values and duplicates

In [0]:
# Remove rows with NaN values
df_final = df_final.dropna()

# Remove duplicate rows
df_final = df_final.drop_duplicates()


##Verifying for logical outliers not addressed by the IQR

In [0]:
# Identify and count the rows to be filtered out
rows_to_remove = df_final[(df_final['o2sat'] > 100) | (df_final['o2sat'] < 0)].shape[0]
# Apply the filtering condition
df_final = df_final[~((df_final['o2sat'] > 100) | (df_final['o2sat'] < 0))]
# Identify the rows to be dropped
rows_to_drop = df_final[df_final['resprate'] < 0].index
# Drop the rows
df_final.drop(rows_to_drop, axis=0, inplace=True)
# Apply the filtering condition
df_final = df_final[~((df_final['heartrate'] > 400) | (df_final['heartrate'] < 30))]
df_final = df_final[~((df_final['temperature'] > 109) | (df_final['temperature'] < 90))]

In [0]:
df_final.shape

##Excluding physically impossible scenarios: Systolic blood pressure less than or equal to diastolic blood pressure

In [0]:
# Define the condition that removes rows with an impossible condition where sbp is less than or equal to dbp
valid_bp_condition = df_final['sbp'] > df_final['dbp']

# Filter the DataFrame - keep all rows that meet the condition
df_final = df_final[valid_bp_condition]

# Reset the index after filtering
df_final = df_final.reset_index(drop=True)

In [0]:
df_final.shape

##Feature Engineering

In [0]:
df_final['pulse_pressure'] = df_final['sbp'] - df_final['dbp']  # Difference between systolic and diastolic blood pressure
df_final['oxygen_ratio'] = df_final['o2sat'] / df_final['resprate']  # Ratio of oxygen saturation to respiratory rate
# Examples of interaction features
df_final['temp_pulse_pressure'] = df_final['temperature'] * df_final['pulse_pressure']  # Interaction between temperature and pulse pressure

In [0]:
# Check and handle NaN and inf values
df_final.replace([np.inf, -np.inf], np.nan, inplace=True)  # Replace inf with NaN
df_final.dropna(inplace=True)  # Remove rows with NaN

##Scaling and splitting sata into training and test sets



In [0]:
df_final = df_final.reset_index(drop=True)

In [0]:
df_final = df_final.drop(columns='index')

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df_final.columns = df_final.columns.astype(str)

# Check for the 'acuity' class (target variable)
if 'acuity' in df_final.columns:
    target_counts = df_final['acuity'].value_counts()
    print(target_counts)
    # Techniques for balancing can be used if there is a significant imbalance

# Columns to be scaled
columns_to_scale = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp']

# Initialize the scaler
scaler = StandardScaler()

# Apply the scaler only to the specified columns
df_final[columns_to_scale] = scaler.fit_transform(df_final[columns_to_scale])

In [0]:
!pip install pycaret

In [0]:
from pycaret.datasets import get_data
from pycaret.classification import *


# Postavljanje okruženja
s = setup(df_final, target='acuity')

# Treniranje i odabir najboljeg modela
best_model = compare_models()


In [0]:
# Separate features and the target variable
X = df_final.drop('acuity', axis=1)
y = df_final['acuity']

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##NN-Sequential

In [0]:
pip install tensorflow

In [0]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [0]:
from sklearn.preprocessing import LabelBinarizer
lb = LabelBinarizer()
y_one_hot = lb.fit_transform(y)

# Podjela podataka
X_train, X_test, y_train, y_test = train_test_split(X, y_one_hot, test_size=0.2, random_state=42)

# Definiranje modela
model = Sequential()
model.add(Dense(units=64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(units=32, activation='relu'))
model.add(Dense(units=5, activation='softmax'))  # Izlazni sloj za 5 klasa

# Kompilacija modela
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Treniranje modela
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2)

# Evaluacija modela
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Predviđanja
predictions = model.predict(X_test)

# Ako želite vidjeti klase s najvišom vjerojatnošću za svako predviđanje
predicted_classes = predictions.argmax(axis=-1)
print(predicted_classes)


In [0]:
from sklearn.metrics import classification_report
import numpy as np

# Provjerite oznake klasa i pretvorite ih u stringove
class_names = [str(int(cls)) for cls in lb.classes_]
print("Klase:", class_names)

# Pretvaranje y_test iz one-hot kodiranog oblika u klasu indeksa
y_test_classes = np.argmax(y_test, axis=-1)

# Predviđanje klasa za testne podatke
predicted_classes = predictions.argmax(axis=-1)

# Ispis preciznosti za sve klase
report = classification_report(y_test_classes, predicted_classes, target_names=class_names, zero_division=0)
print(report)


##Models: Selection based on correlation for predictive modeling and removing bias from the training set

In [0]:
# Separate features and the target variable
X = df_final.drop('acuity', axis=1)
y = df_final['acuity']

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

RandomForest

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, accuracy_score, roc_curve, auc,  recall_score, f1_score

# Initialize the model
model = RandomForestClassifier(random_state=42, class_weight={1: 1, 2: 1, 3: 1, 4: 1,5:1})
# Train the model on the resampled data
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test, y_pred, average=None)[0]
print("Precision for class 1:", precision_class_1)

In [0]:
# Compute probabilities for all classes
y_proba = model.predict_proba(X_test)

# Find indices of misclassified samples for class 1
misclassified_indices = np.where((y_test == 1) & (y_pred != 1))[0]

# Select the first 10 misclassified samples
top_misclassified_indices = misclassified_indices[:10]

# Filter probabilities for the first 10 misclassified samples
top_misclassified_proba = y_proba[top_misclassified_indices]

# Create a DataFrame for easier handling
# Assuming classes are from 1 to 5
top_misclassified_df = pd.DataFrame(top_misclassified_proba, columns=[f'Class {i}' for i in range(1, y_proba.shape[1] + 1)])
top_misclassified_df['True Label'] = y_test.iloc[top_misclassified_indices].values
top_misclassified_df['Predicted Label'] = y_pred[top_misclassified_indices]

# Display the first 10 rows
print(top_misclassified_df)

# Plot for the first 10 misclassified samples
plt.figure(figsize=(12, 8))

for i, row in top_misclassified_df.iterrows():
    plt.plot(row.index[:-2], row[:-2], marker='o', label=f'Instance {i} - True: {int(row["True Label"])} Pred: {int(row["Predicted Label"])}')

plt.xlabel('Class')
plt.ylabel('Probability')
plt.title('Probabilities for Misclassified Samples (Top 10)')
plt.legend(loc='best')
plt.grid(True)
plt.xticks(ticks=np.arange(y_proba.shape[1]), labels=[f'Class {i}' for i in range(1, y_proba.shape[1] + 1)])
plt.show()

In [0]:
# Compute probabilities for all classes
y_proba = model.predict_proba(X_test)

# Find indices of misclassified samples for class 1
misclassified_indices = np.where((y_test == 1) & (y_pred != 1))[0]

# Select the first 10 misclassified samples
top_misclassified_indices = misclassified_indices[:10]

# Filter probabilities for the first 10 misclassified samples
top_misclassified_proba = y_proba[top_misclassified_indices]

# Create a DataFrame for easier handling
top_misclassified_df = pd.DataFrame(top_misclassified_proba, columns=[f'Class {i}' for i in range(1, y_proba.shape[1] + 1)])
top_misclassified_df['True Label'] = y_test.iloc[top_misclassified_indices].values
top_misclassified_df['Predicted Label'] = y_pred[top_misclassified_indices]

# Display the first 10 rows
print(top_misclassified_df)

# Histogram plot for the first 10 misclassified samples
plt.figure(figsize=(12, 8))

# Define the width of bars
bar_width = 0.8 / len(top_misclassified_indices)

# Iterate through each of the first 10 instances
for i, row in top_misclassified_df.iterrows():
    # Set x positions for bars
    x = np.arange(len(row) - 2) + i * bar_width
    plt.bar(
        x,
        row[:-2],
        width=bar_width,
        label=f'Instance {i} - True: {int(row["True Label"])} Pred: {int(row["Predicted Label"])}',
        alpha=0.6
    )

plt.xlabel('Class')
plt.ylabel('Probability')
plt.title('Probabilities for Misclassified Samples (Top 10)')
plt.xticks(ticks=np.arange(y_proba.shape[1]), labels=[f'Class {i}' for i in range(1, y_proba.shape[1] + 1)])
plt.legend(loc='best', bbox_to_anchor=(1.05, 1))
plt.grid(True)
plt.tight_layout()
plt.show()


In [0]:
# Compute probabilities for all classes
y_proba = model.predict_proba(X_test)

# Find indices of misclassified samples for class 1
misclassified_indices = np.where((y_test == 1) & (y_pred != 1))[0]

# Select the first 10 misclassified samples
top_misclassified_indices = misclassified_indices[:10]

# Filter probabilities for the first 10 misclassified samples
top_misclassified_proba = y_proba[top_misclassified_indices]

# Create a DataFrame for easier handling
top_misclassified_df = pd.DataFrame(top_misclassified_proba, columns=[f'Class {i}' for i in range(1, y_proba.shape[1] + 1)])
top_misclassified_df['True Label'] = y_test.iloc[top_misclassified_indices].values
top_misclassified_df['Predicted Label'] = y_pred[top_misclassified_indices]

# Display the first 10 rows
print(top_misclassified_df)

# Create a figure with 10 subplots
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(15, 20))
axes = axes.flatten()  # Makes indexing the subplots easier

# Plot histograms for the first 10 misclassified samples
for i, (index, row) in enumerate(top_misclassified_df.iterrows()):
    axes[i].bar(
        x=np.arange(len(row) - 2),
        height=row[:-2],
        color='blue',
        alpha=0.7
    )
    axes[i].set_title(f'Instance {i} - True: {int(row["True Label"])} Pred: {int(row["Predicted Label"])}')
    axes[i].set_xticks(np.arange(len(row) - 2))
    axes[i].set_xticklabels([f'Class {j}' for j in range(1, y_proba.shape[1] + 1)], rotation=45)
    axes[i].set_xlabel('Class')
    axes[i].set_ylabel('Probability')
    axes[i].grid(True)

plt.tight_layout()
plt.show()

In [0]:
# Calculate the classification report
report = classification_report(y_test, y_pred, output_dict=True, target_names=['Class 1', 'Class 2', 'Class 3', 'Class 4', 'Class 5'])

# Prepare data for visualization
metrics = ['precision', 'recall', 'f1-score']
classes = ['Class 1', 'Class 2', 'Class 3', 'Class 4', 'Class 5']
data = {metric: [report[cls][metric] for cls in classes] for metric in metrics}

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8))
x = range(len(classes))
width = 0.2

for i, metric in enumerate(metrics):
    ax.bar([p + width*i for p in x], data[metric], width=width, label=metric)

ax.set_xlabel('Classes')
ax.set_ylabel('Scores')
ax.set_title('Classification Metrics')
ax.set_xticks([p + width for p in x])
ax.set_xticklabels(classes)
ax.legend()

plt.show()


In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

# Assuming y_test and y_pred are your test labels and predictions

# Print accuracy score
print("Accuracy:", accuracy_score(y_test, y_pred))

# Get the detailed classification report
report = classification_report(y_test, y_pred, output_dict=True)
report_str = classification_report(y_test, y_pred)

# Create a figure and axis
fig, ax = plt.subplots(figsize=(10, 2))  # Adjust the size as needed

# Add the classification report as text in a green box, shifted slightly to the left
report_text = f"Classification Report:\n\n{report_str}"
plt.text(0.3, 0.5, report_text, fontsize=12, ha='left', va='center',
         bbox=dict(facecolor='lightgreen', edgecolor='black', boxstyle='round,pad=1'))

# Hide the axis
plt.axis('off')

# Display the plot
plt.show()

# Extract all classes
classes = [label for label in report.keys() if label.isdigit()]  # Assuming class labels are numeric

# Create a DataFrame for visualization
for cls in classes:
    class_metrics = report[cls]

    metrics_df = pd.DataFrame({
        'Metric': ['Precision', 'Recall', 'F1-Score', 'Support'],
        'Value': [class_metrics['precision'], class_metrics['recall'], class_metrics['f1-score'], class_metrics['support']]
    })

    # Plot the metrics with colored bars
    plt.figure(figsize=(8, 5))
    colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFD700']  # Different colors for different metrics
    sns.barplot(x='Metric', y='Value', data=metrics_df, palette=colors)
    plt.title(f'Metrics for Class {cls}')
    plt.ylabel('Value')
    plt.xlabel('Metric')
    plt.ylim(0, 1.1)  # To have a clear view
    plt.show()

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, classification_report

# Assuming y_test and y_pred are your test labels and predictions

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision_class_1 = precision_score(y_test, y_pred, labels=[1], average='macro')
report = classification_report(y_test, y_pred)

# Set the style for the plot
sns.set(style="whitegrid")

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Define green shades
light_green = '#98fb98'      # PaleGreen
medium_green = '#90ee90'     # LightGreen
lightest_green = '#f0fff0'   # Honeydew

# Display results in colored boxes with green shades
ax.text(0.05, 0.75, f'Overall Accuracy: {accuracy:.2f}', fontsize=12, bbox=dict(facecolor=light_green, alpha=0.6))
ax.text(0.05, 0.55, f'Precision for class 1: {precision_class_1:.2f}', fontsize=12, bbox=dict(facecolor=light_green, alpha=0.6))
ax.text(0.05, 0.1, f'Classification Report:\n{report}', fontsize=10, bbox=dict(facecolor=light_green, alpha=0.6), ha='left', va='top')

# Remove axes and add title
ax.axis('off')
ax.set_title("Model Evaluation Results", fontsize=16, pad=20)

# Show the plot
plt.tight_layout()
plt.show()


In [0]:
from sklearn.preprocessing import label_binarize


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)  # Get the predicted probabilities


# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test, y_pred, average=None)[0]
print("Precision for class 1:", precision_class_1)

# Binarize the output labels for ROC curve computation
y_test_bin = label_binarize(y_test, classes=[1, 2, 3, 4, 5])
n_classes = y_test_bin.shape[1]

# Compute ROC curve and ROC area for each class
fpr = {}
tpr = {}
roc_auc = {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure()
colors = ['blue', 'red', 'green', 'orange', 'purple']
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'Class {i+1} (area = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.show()


We see that class 1 is mostly mixed with class 2, so let's try dividing our data into two classes: 'urgent' and 'non-urgent.

In [0]:
def transform_labels(y):
    return np.where(np.isin(y, [1, 2]), 'urgent', 'non-urgent')

# Transform the target variables
y_test_transformed = transform_labels(y_test)
y_pred_transformed = transform_labels(y_pred)

# Calculate metrics
print(classification_report(y_test_transformed, y_pred_transformed))
print(confusion_matrix(y_test_transformed, y_pred_transformed))

# Calculate precision for 'urgent'
precision_urgent = precision_score(y_test_transformed, y_pred_transformed, pos_label='urgent')
print("Precision for class 'urgent':", precision_urgent)

# Prepare data for ROC curve
# Convert labels to binary format for ROC calculation
y_test_bin = label_binarize(y_test_transformed, classes=['non-urgent', 'urgent'])
y_pred_prob = model.predict_proba(X_test)[:, 1]  # Get probabilities for the 'urgent' class

# Compute ROC curve and ROC area
fpr, tpr, _ = roc_curve(y_test_bin, y_pred_prob)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.show()


XGBoost

In [0]:
pip install xgboost

In [0]:
import xgboost as xgb

# Convert y_train and y_test to integer labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Create the XGBoost model
model = xgb.XGBClassifier(
    objective='multi:softmax',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

In [0]:
# Make predictions on the test set
y_pred = model.predict(X_test)

In [0]:
# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test, y_pred, average=None)[0]
print("Precision for class 1:", precision_class_1)

Selection based on correlation for predictive modeling

In [0]:
# Separate features and the target variable
X = df_final.drop('acuity', axis=1)  # Replace 'acuity' with the actual target variable if different
y = df_final['acuity']

# Find the absolute values of the correlations between each feature and the target variable
correlations = X.apply(lambda x: abs(x.corr(y)))

# Sort the correlations and find the optimal k
sorted_features = correlations.sort_values(ascending=False)
optimal_k = 0
max_correlation_sum = 0

# Find the k that provides the highest sum of absolute correlation values
for k in range(1, len(sorted_features) + 1):
    correlation_sum = sorted_features[:k].sum()
    if correlation_sum > max_correlation_sum:
        max_correlation_sum = correlation_sum
        optimal_k = k

# Store the selected features in selected_features
selected_features = sorted_features[:optimal_k].index.tolist()

# Print the optimal k and the selected features
print(f"Optimal k: {optimal_k}")
print(f"Selected features: {selected_features}")

# Print the absolute correlation values for the selected features
print("Absolute correlation values for the selected features:")
print(sorted_features[:optimal_k])


In [0]:
# Selected features
selected_features = sorted_features[:optimal_k].index.tolist()

# Create a new DataFrame containing only the selected features
X_selected = df_final[selected_features]
y = df_final['acuity']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

# Check the shapes of the created sets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

RandomForest with selected features

In [0]:
# Initialize the model
model = RandomForestClassifier(random_state=42, class_weight={1: 1, 2: 1, 3: 1, 4: 1,5:1})
# Train the model on the resampled data
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test, y_pred, average=None)[0]
print("Precision for class 1:", precision_class_1)

Removing bias from train set



In [0]:
from imblearn.over_sampling import SMOTE

# Check class distribution before balancing
print("Class distribution before balancing:")
print(y_train.value_counts())

# Apply SMOTE for oversampling
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Check class distribution after balancing
print("\nClass distribution after balancing:")
print(pd.Series(y_train).value_counts())


In [0]:
y_train.unique()

Randomforest with removed bias from train set

In [0]:
# Initialize the model
model = RandomForestClassifier(random_state=42, class_weight={1: 1, 2: 1, 3: 1, 4: 1,5:1})
# Train the model on the resampled data
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test, y_pred, average=None)[0]
print("Precision for class 1:", precision_class_1)

Randomforest with removed bias from train set and with weight 2 for class 1

In [0]:
# Initialize the model with a focus on precision for class 1
model = RandomForestClassifier(random_state=42, class_weight={1: 2, 2: 1, 3: 1, 4: 1,5:1})  # Adjust weights as needed

# Train the model on the resampled data
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test, y_pred, average=None)[0]
print("Precision for class 1:", precision_class_1)

Let's try LightGBM, CatBoost, and XGBoost

In [0]:
pip install lightgbm

In [0]:
import lightgbm as lgb

# Initialize the LightGBM model with a focus on precision for class 1
lgb_model = lgb.LGBMClassifier(class_weight={1: 1, 2: 1, 3: 1, 4: 1, 5:1}, random_state=42)

# Train the model on the resampled data
lgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_lgb = lgb_model.predict(X_test)

# Evaluate the model
print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgb))
print("LightGBM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgb))

# Calculate precision specifically for class 1
precision_class_1_lgb = precision_score(y_test, y_pred_lgb, average=None)[0]
print("Precision for class 1 (LightGBM):", precision_class_1_lgb)


Catboost

In [0]:
pip install catboost

In [0]:
from catboost import CatBoostClassifier

# Initialize the CatBoost model with a focus on precision for class 1
catboost_model = CatBoostClassifier(class_weights={1: 1, 2: 1, 3: 1, 4: 1, 5: 1}, random_state=42, verbose=0)

# Train the model on the resampled data
catboost_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_catboost = catboost_model.predict(X_test)

# Evaluate the model
print("CatBoost Classification Report:")
print(classification_report(y_test, y_pred_catboost))
print("CatBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_catboost))

# Calculate precision specifically for class 1
precision_class_1_catboost = precision_score(y_test, y_pred_catboost, average=None)[0]
print("Precision for class 1 (CatBoost):", precision_class_1_catboost)


XGBoost

In [0]:
pip install xgboost

In [0]:
import xgboost as xgb

# Inicijalizujte LabelEncoder
label_encoder = LabelEncoder()

# Kodirajte ciljne varijable
y_train_resampled_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Inicijalizujte XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')

# Obučite model na resampliranim podacima
xgb_model.fit(X_train, y_train_resampled_encoded)

# Napravite predikcije na test skupu
y_pred_xgb_encoded = xgb_model.predict(X_test)

# Dekodirajte predikcije
y_pred_xgb = label_encoder.inverse_transform(y_pred_xgb_encoded)

# Evaluirajte model
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

# Izračunajte preciznost posebno za klasu 1
# Ako su klase [1, 2, 3, 4], klasa 1 je indeks 0 u mapiranom formatu
precision_class_1_xgb = precision_score(y_test, y_pred_xgb, labels=[1], average=None)[0]
print("Precision for class 1 (XGBoost):", precision_class_1_xgb)

Now we will replace SMOTE with stratified sampling

In [0]:
from sklearn.model_selection import StratifiedShuffleSplit

# Define features and target variable
X = df_final.drop('acuity', axis=1)
y = df_final['acuity']

# Create a StratifiedShuffleSplit object
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

# Split the data using stratified sampling
for train_index, test_index in sss.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Check the class distribution in the training and test sets
print("Class distribution in the training set:")
print(y_train.value_counts(normalize=True))

print("\nClass distribution in the test set:")
print(y_test.value_counts(normalize=True))


In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [0]:
from sklearn.feature_selection import SelectKBest, f_classif

# Definišite broj karakteristika koje želite da odaberete (ovde koristimo optimal_k koji ste prethodno izračunali)
selector = SelectKBest(score_func=f_classif, k=optimal_k)

# Primenite odabir karakteristika na obučni skup
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)


Logistic Regression, RandomForest and Grid Search

Logistic regression on scaled data with all variables.

In [0]:
from sklearn.linear_model import LogisticRegression

# Create and train the model
model = LogisticRegression(random_state=42)
model.fit(X_train_selected, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_selected)


In [0]:
# Print the evaluation metrics
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


RandomForest with all variables.

In [0]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)


In [0]:
# Performance Evaluation
print("Classification Report for Random Forests:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Accuracy
print("Random Forest Model Accuracy:", accuracy_score(y_test, y_pred))


Gridsearch for RandomForest.

In [0]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for the search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize GridSearchCV with the Random Forest model
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           cv=3,  # 3-fold cross-validation
                           n_jobs=-1,  # Parallel execution
                           verbose=2)

# Train the model with optimal hyperparameters
grid_search.fit(X_train, y_train)

# Print the best parameters and performance
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

# Use the best model for prediction
best_rf_model = grid_search.best_estimator_
y_pred_best = best_rf_model.predict(X_test)

# Evaluate the best model
print("Classification Report for Optimized Random Forest Model:")
print(classification_report(y_test, y_pred_best))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("Accuracy of Optimized Random Forest Model:", accuracy_score(y_test, y_pred_best))

RandomForest with all scaled variables.

In [0]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)


In [0]:
# Performance Evaluation
print("Classification Report for Random Forests:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Accuracy
print("Accuracy of the Random Forest Model:", accuracy_score(y_test, y_pred))


RandomForest with selected and scaled variables.

In [0]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train_selected, y_train)

y_pred = rf_model.predict(X_test_selected)


In [0]:
# Performance Evaluation
print("Classification Report for Random Forests:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Accuracy
print("Accuracy of the Random Forest Model:", accuracy_score(y_test, y_pred))

#Training Dynamics: For diagnosing and improving the training rocess

Let's proceed with XGBoost, which has emerged as our most effective model to date. We will specifically focus on evaluating the precision of Class 1, as it is our primary measure of performance. Then, we will try to enhance the model with training dynamics to further improve its performance.

In [0]:
X = df_final.drop(columns=['acuity'])
y = df_final['acuity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
le = LabelEncoder()

# Combine all target variables into a single array for encoding
combined_y = np.concatenate((y_train, y_test))

# Fit and transform all target variables
le.fit(combined_y)

# Transform y_train and y_test
y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [0]:
base_model = xgb.XGBClassifier(
    objective='multi:softmax',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42
)
base_model.fit(X_train, y_train)

In [0]:
y_test_pred = base_model.predict(X_test)
y_test_pred_proba = base_model.predict_proba(X_test)

In [0]:
# Import necessary metrics and visualization libraries
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# Print accuracy score
print("Accuracy:", accuracy_score(y_test, y_test_pred))

# Print the detailed classification report
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))

# Generate and display the confusion matrix
conf_matrix = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=base_model.classes_, yticklabels=base_model.classes_)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

First, we will focus on incorporating difficult examples (based on log loss) to enhance the model's learning capability. After that, we will address and remove any potential noise in the data.

Predict probabilities for the training set and calculate log loss per sample.

In [0]:
from sklearn.metrics import log_loss
y_train_pred_proba = base_model.predict_proba(X_train)
train_log_loss = log_loss(y_train, y_train_pred_proba, labels=base_model.classes_, sample_weight=None, normalize=False)

Add the calculated log loss back to the corresponding rows in the original DataFrame.

In [0]:
# Convert X_train and y_train to DataFrame if not already
if isinstance(X_train, np.ndarray):
    X_train = pd.DataFrame(X_train, columns=X.columns)

if isinstance(y_train, np.ndarray):
    y_train = pd.Series(y_train)

# Combine X_train and y_train into a single DataFrame
train_df = X_train.copy()
train_df['acuity'] = y_train.values

# Display the first few rows of the combined DataFrame
train_df.head()

In [0]:
train_df['train_loss']=train_log_loss

In [0]:
train_df.head()

Identify examples with the highest log loss (top 10% most difficult cases).


In [0]:
loss_threshold = np.percentile(train_df['train_loss'].dropna(), 90)  # e.g., the top 10% most difficult cases
hard_examples = train_df[train_df['train_loss'] > loss_threshold]

sorted_probs = np.sort(y_train_pred_proba, axis=1)
margin = sorted_probs[:, -1] - sorted_probs[:, -2]
margin_threshold = np.percentile(margin, 20)  # Manji margin znači teži slučaj

# Maksimalna verovatnoća za svaku instancu
max_probs = np.max(y_train_pred_proba, axis=1)
uncertainty = 1 - max_probs
uncertainty_threshold = np.percentile(uncertainty, 80)

# Neizvesnost je 1 - max_probs
uncertainty = 1 - max_probs
# Identifikuj teške primere (visok loss ILI visoka neizvesnost ILI mali margin)
hard_examples = train_df[
    (train_log_loss > loss_threshold) |
    (uncertainty > uncertainty_threshold) |
    (margin < margin_threshold)
]


Oversample the difficult examples to balance the dataset.

In [0]:
df_balanced = pd.concat([train_df, hard_examples])

Detect potential noise in labels using Isolation Forest on the balanced dataset.

In [0]:
from sklearn.ensemble import IsolationForest
iso_forest = IsolationForest(contamination=0.001, random_state=42)
df_balanced['anomaly_score'] = iso_forest.fit_predict(df_balanced.drop(columns=['acuity', 'train_loss']))

Filter potential anomalies from df_balanced.

In [0]:
cleaned_df = df_balanced[df_balanced['anomaly_score'] == 1]

Train a model with the improved dataset.

In [0]:
X_train_cleaned = cleaned_df.drop(columns=['acuity', 'train_loss', 'anomaly_score'])
y_train_cleaned = cleaned_df['acuity']

final_model = xgb.XGBClassifier(
    objective='multi:softmax',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42
)

final_model.fit(X_train_cleaned, y_train_cleaned)

Evaluate the new model on the test set.

In [0]:
y_test_pred = final_model.predict(X_test)
y_test_pred_proba = final_model.predict_proba(X_test)

In [0]:
# Import necessary metrics and visualization libraries
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# Print accuracy score
print("Accuracy:", accuracy_score(y_test, y_test_pred))

# Print the detailed classification report
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))

# Generate and display the confusion matrix
conf_matrix = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=final_model.classes_, yticklabels=final_model.classes_)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

Training Dynamics Results:

After applying training dynamics, including incorporating difficult examples and refining the model, we observed that the results did not improve as expected. In fact, there was a slight deterioration in performance. This outcome suggests that while the intent was to enhance the model's ability to learn from challenging examples, it may have introduced additional noise or complexity that affected the model's generalization ability.

It is important to consider the following:

Complexity of the Task: The complexity of the data or the task may have increased, leading to a decrease in performance. Balancing challenging examples with clean data is crucial.

Potential Overfitting: Introducing difficult examples or modifications may have led to overfitting, where the model performs well on the training data but poorly on unseen data.

Data Quality: Ensuring high-quality data is essential. Any noise or errors in the data could impact the model's performance negatively.

Further adjustments may be necessary, such as refining the selection of difficult examples or revisiting preprocessing steps, to better align the model with the underlying patterns in the data.

#Second method for addressing the problem

Now let's try a different approach to data editing and apply bias removal with ADASYN on all the data. Keep in mind that this may potentially reduce the ability to generalize to new data.

##Data loading

In [0]:
df1 = pd.read_csv('/content/drive/MyDrive/data/edstays.csv')
df2 = pd.read_csv('/content/drive/MyDrive/data/triage.csv')
df1 = df1.drop(columns=['stay_id'])
df = pd.merge(df1, df2, on='subject_id')
df.head()

In [0]:
df.shape

In [0]:
# Check for duplicate column names
duplicate_names = df.columns[df.columns.duplicated()]

# Create a new list of column names with a prefix added for duplicate names
new_names = []
counter = {}
for column in df.columns:
    if column in duplicate_names:
        if column not in counter:
            counter[column] = 1
        else:
            counter[column] += 1
        new_name = f"{column}_{counter[column]}"
    else:
        new_name = column
    new_names.append(new_name)

# Update the DataFrame with new column names
df.columns = new_names

# Print new column names for verification
df.columns

In [0]:
df.shape

##Filtering columns based on correlation, for which we needed to encode the columns by creating a categorical mapping.

In [0]:
df.info()

In [0]:
# Iterate through all columns in the DataFrame
for column in df.columns:
    # Check if the column is non-numeric
    if df[column].dtype == 'object':
        # Apply pd.factorize to non-numeric columns
        df[column], unique_ids = pd.factorize(df[column])
        print(f'Column: {column}, Unique IDs: {unique_ids}')


In [0]:
df.info()

In [0]:
# Check which columns need to be encoded
non_numeric_columns = df.select_dtypes(include=['object', 'category']).columns

# Initialize LabelEncoder
encoders = {}

# Encode categorical data into numerical values
for column in non_numeric_columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column].astype(str))
    encoders[column] = le

# Check if the 'acuity' column exists
if 'acuity' in df.columns:
    # Calculate the correlation between all columns and the 'acuity' column
    correlations = df.corr()['acuity'].abs().sort_values(ascending=False)

    # Select the 20 columns with the highest correlation
    most_correlated = correlations[1:21]  # Skip the first element, which is the correlation with itself

    # Print the results
    for i, (column_name, correlation) in enumerate(most_correlated.items()):
        index = df.columns.get_loc(column_name)
        print(f"{i + 1}. Column: {column_name}, Index: {index}, Correlation: {correlation:.4f}")
else:
    print("The 'acuity' column does not exist in the data.")


In [0]:
# Iterate through all columns in the DataFrame
for column in df.columns:
    # Get unique values of the column
    unique_values = df[column].unique()
    num_unique_values = len(unique_values)

    # Print the number of unique values for the column
    print(f"Column: {column}")
    print(f"Number of unique values: {num_unique_values}")

    # If the number of unique values is less than 10, list them
    if num_unique_values < 10:
        print("Unique values:")
        for value in unique_values:
            print(value)
    print()  # Blank line for better formatting of output


##Let's now select only the most correlated columns with respect to the target column we want to predict.

In [0]:
# List of columns to keep
desired_columns = [
    'subject_id', 'stay_id', 'gender', 'race', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'chiefcomplaint', 'intime', 'arrival_transport', 'acuity'
]

# Select only the desired columns
df = df[desired_columns]

# Print the first few rows to check the results
df.head()


##Unique values in the 'acuity' column that we are predicting, and removal of rows with NaN values in this column.

In [0]:
# Print unique values in the 'acuity' column
unique_values = df['acuity'].unique()

print("Unique values in the 'acuity' column:")
print(unique_values)

In [0]:
# Count the number of rows with NaN values in the 'acuity' column
nan_count_before = df['acuity'].isna().sum()
print(f"Number of rows with NaN in the 'acuity' column before removal: {nan_count_before}")

In [0]:
# Remove rows with NaN values in the 'acuity' column
df = df.dropna(subset=['acuity'])

In [0]:
# Count the number of rows with NaN values in the 'acuity' column after removal
nan_count_after = df['acuity'].isna().sum()
print(f"Number of rows with NaN in the 'acuity' column after removal: {nan_count_after}")

In [0]:
df.shape

##Handling NaN values: Remove rows with more than 30% NaN values, and fill in the remaining NaN values.

In [0]:
# Count NaN values per column
nan_counts = df.isna().sum()

# Print the number of NaN values per column
print("Number of NaN values per column:")
print(nan_counts)

In [0]:
# Print the percentage of NaN values per column
nan_percentage = (df.isna().sum() / df.shape[0]) * 100
print("Percentage of NaN values per column:")
print(nan_percentage)

In [0]:
# Remove rows with NaN values in the 'temperature' column
df = df.dropna(subset=['temperature'])

In [0]:
# Check data types for specific columns
numeric_columns = ['heartrate', 'resprate', 'o2sat', 'sbp', 'dbp']

# Fill numeric columns with the mean value
for column in numeric_columns:
    if df[column].dtype in ['float64', 'int64']:  # Check if the column is numeric
        df[column].fillna(df[column].mean(), inplace=True)

# Print the first few rows after filling
df.head()

In [0]:
# Count NaN values per column
nan_counts = df.isna().sum()

# Print the number of NaN values per column
print("Number of NaN values per column:")
print(nan_counts)

### Check for and remove duplicate columns and rows

In [0]:
df.shape

In [0]:
# Remove duplicate columns from the DataFrame
df = df.loc[:, ~df.columns.duplicated()]

In [0]:
# Count the number of duplicated rows in the DataFrame
duplicates_count = df.duplicated().sum()
print(f"Number of duplicated rows: {duplicates_count}")

# Remove duplicate rows from the DataFrame
df.drop_duplicates(inplace=True)

In [0]:
df.shape

##Removing outliers using a statistical method: Z-score

In [0]:
from scipy import stats
# Calculate Z-scores for each numeric column
z_scores = np.abs(stats.zscore(df.select_dtypes(include=np.number)))

# Identify outliers (rows with Z-scores greater than 3)
outliers = (z_scores > 3).any(axis=1)

# Remove outliers, except rows where 'acuity' equals 5
df = df[~(outliers & (df['acuity'] != 5))]

# Print the number of outliers and the number of rows without outliers
num_outliers = outliers.sum() - (~outliers & (df['acuity'] == 5)).sum()
num_rows_without_outliers = df.shape[0]

print(f"Number of outliers identified using the Z-score method: {num_outliers}")
print(f"Number of rows without outliers: {num_rows_without_outliers}")


##Splitting the dataset into features (X) and target variable (y), and then scaling the features.

In [0]:
X=df.drop("acuity",axis=1)
y=df["acuity"]

In [0]:
X= scaler.fit_transform(X)

##Removing bias before splitting the data into training and test sets.

In [0]:
from imblearn.over_sampling import ADASYN

# Check the number of instances per class
adasyn = ADASYN(random_state=42)
X, y = adasyn.fit_resample(X, y)

# Check the class distribution after oversampling
print("\nClass distribution after oversampling:")
print(pd.Series(y).value_counts())


In [0]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the number of rows in each set
print(f"Number of rows in X_train: {X_train.shape[0]}")
print(f"Number of rows in X_test: {X_test.shape[0]}")
print(f"Number of rows in y_train: {y_train.shape[0]}")
print(f"Number of rows in y_test: {y_test.shape[0]}")

In [0]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions on the test set
    predictions = model.predict(X_test)

    # Calculate various metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, average='weighted')
    recall = recall_score(y_test, predictions, average='weighted')
    f1 = f1_score(y_test, predictions, average='weighted')
    cm = confusion_matrix(y_test, predictions)

    # Print the results
    print(f'Accuracy of model {model.__class__.__name__}: {accuracy:.4f}')
    print(f'Precision of model {model.__class__.__name__}: {precision:.4f}')
    print(f'Recall of model {model.__class__.__name__}: {recall:.4f}')
    print(f'F1-score of model {model.__class__.__name__}: {f1:.4f}')
    print(f'Confusion matrix for model {model.__class__.__name__}:\n{cm}')


In [0]:
rf = RandomForestClassifier(random_state=357)
evaluate_model(rf, X_train, y_train, X_test, y_test)

#Third method for addressing the problem

We manipulated the data by removing what we consider impossible cases. Specifically, we categorized chiefcomplaints that belong to class 1 and have all vital signs normal as inherently indicative of class 1. Consequently, any instances with these characteristics that belong to other classes have been removed.

It is important to note that, similar to Method 2, this approach could potentially reduce the model's ability to generalize to new data. Caution is advised, as these adjustments were made solely for the purpose of analysis.

##Data loading and exploratory analysis

In [0]:
df1 = pd.read_csv('triage.csv')
df2=pd.read_csv('edstays.csv')
df = pd.merge(df1, df2, on='stay_id', how='left')

In [0]:
columns_to_keep = [
    'subject_id_x',
    'stay_id',
    'temperature',
    'heartrate',
    'resprate',
    'o2sat',
    'sbp',
    'dbp',
    'pain',
    'acuity',
    'gender',
    'arrival_transport',
    'chiefcomplaint',
    'race'
]

df = df[columns_to_keep]
df = df.dropna()
df.head()

In [0]:
df.tail()

In [0]:
df.describe()

##Explore column by column (outliers, mapping)


###temperature

In [0]:
df['temperature'].value_counts()

This code creates a new column called temperature_celsius in the DataFrame df, which contains the converted temperature values in degrees Celsius.

In [0]:
# Convert Fahrenheit to Celsius
df['temperature_celsius'] = (df['temperature'] - 32) * 5/9

In [0]:
# Create a boxplot to visualize outliers
sns.boxplot(x=df['temperature_celsius'])
plt.show()

In [0]:
df['temperature_celsius'].value_counts()

In [0]:
# Count values below 32°C
below_32 = df[df['temperature_celsius'] < 32].shape[0]

# Count values above 42°C
above_50 = df[df['temperature_celsius'] > 42].shape[0]

# Print the results
print(f"Number of values below 21°C: {below_32}")
print(f"Number of values above 42°C: {above_50}")

In [0]:
below_32 = df[df['temperature_celsius']< 32]
above_50 = df[df['temperature_celsius'] > 42]


print("Temperature below 32°C and 'acuity':")
print(below_32[['temperature_celsius', 'acuity']])

print("Temperature above 50°C and 'acuity':")
print(above_50[['temperature_celsius', 'acuity']])

In [0]:
# Remove rows where temperature is below 32°C or above 42°C
df = df[(df['temperature_celsius'] >= 32) & (df['temperature_celsius'] <= 42)]

# Display the cleaned DataFrame
df

In [0]:
df = df.drop(columns=['temperature'])

In [0]:
def classify_temperature(temp):
    if temp < 27.9:
        return 2
    elif 27.9 <= temp < 36.0:
        return 1
    elif 36.0 <= temp <= 37.2:
        return 0
    elif 37.3 <= temp <= 38.0:
        return 1
    elif 38.1 <= temp <= 39.0:
        return 2
    elif 39.1 <= temp <= 40.0:
        return 3
    else:
        return 4

# Primjena funkcije za klasifikaciju
df['temperature1'] = df['temperature_celsius'].apply(classify_temperature)

###resprate

In [0]:
df['resprate'].value_counts()

In [0]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['resprate'])
plt.title('Boxplot of Respiratory Rate')
plt.xlabel('Respiratory Rate (breaths per minute)')
plt.grid(True)
plt.show()

In [0]:
count_less_than_8 = df[df['resprate'] < 0].shape[0]

# Count the number of rows where respirate is greater than 90
count_greater_than_70 = df[df['resprate'] > 70].shape[0]

# Output the counts
print(f"Number of respirate values less than 8: {count_less_than_8}")
print(f"Number of respirate values greater than 70: {count_greater_than_70}")

In [0]:
df = df[(df['resprate'] >= 0) & (df['resprate'] <= 70)]

In [0]:
def classify_respiratory_rate(rate):

    if rate < 8:
        return 3
    elif rate <= 12:
        return 1
    elif rate <= 20:
        return 0
    elif rate <= 25:
        return 1
    elif rate <= 40:
        return 2
    else:
        return 3

# Apply the classification function to create a new column
df['resprate1'] = df['resprate'].apply(classify_respiratory_rate)

###heartrate

In [0]:
df['heartrate'].value_counts()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x=df['heartrate'])
plt.title('Boxplot of Heart Rate')
plt.xlabel('Heart Rate (bpm)')
plt.grid(True)
plt.show()

In [0]:
# Count values below 0
below_0 = df[df['heartrate'] < 30].shape[0]

# Count values above 400
above_400 = df[df['heartrate'] > 180].shape[0]

# Print the results
print(f"Number of values below 0 bpm: {below_0}")
print(f"Number of values above 400 bpm: {above_400}")

In [0]:
below_30 = df[df['heartrate']< 30]
above_200 = df[df['heartrate'] > 200]


print("Heartrate below 30 and 'acuity':")
print(below_30[['heartrate', 'acuity']])

print("heartrate above 200 and 'acuity':")
print(above_200[['heartrate', 'acuity']])

In [0]:
# Filter the DataFrame to remove rows where heart rate is below 0 or above 400
df = df[(df['heartrate'] >= 0) & (df['heartrate'] <= 400)]

In [0]:
def classify_heart_rate(rate):
    if rate < 40:
        return 2
    elif 40 <= rate < 60:
        return 1
    elif 60 <= rate <= 100:
        return 0
    elif 100 < rate <= 140:
        return 2
    else:
        return 3

# Primjena funkcije za klasifikaciju
df['heartrate1'] = df['heartrate'].apply(classify_heart_rate)

###o2sat

In [0]:
df['o2sat'].value_counts()

In [0]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['o2sat'])
plt.title('Boxplot of Oxygen Saturation')
plt.xlabel('Oxygen Saturation (%)')
plt.grid(True)
plt.show()

In [0]:
# Count values below 0%
below_zero_count = df[df['o2sat'] < 0].shape[0]

# Count values above 100%
above_hundred_count = df[df['o2sat'] > 100].shape[0]

# Print the results
print(f"Number of values below 0%: {below_zero_count}")
print(f"Number of values above 100%: {above_hundred_count}")

In [0]:
df = df[(df['o2sat'] >= 0) & (df['o2sat'] <= 100)]

###pain

In [0]:
df['pain'].value_counts().head(20)

In our dataset, the pain column contains various values. We are only interested in keeping rows where the pain level is between 0 and 10 (inclusive). We will filter the dataset accordingly and discard rows with pain values outside this range.

In [0]:
# Convert all entries in the 'pain' column to lowercase
df['pain'] = df['pain'].str.lower()

# Assign specific numeric values to coded pain levels
df['pain'] = df['pain'].replace({
    'uta': 15,
    'u/a': 15,
    'ua': 15,
    'unable':15,
    'critical': 14
})

# Convert 'pain' column to numeric, coercing errors to NaN
df['pain'] = pd.to_numeric(df['pain'], errors='coerce')

# Filter rows where 'pain' is between 0 and 13 (inclusive), or 14, or 15
df = df[(df['pain'] >= 0) & (df['pain'] <= 15)]

# Round 'pain' values to the nearest integer (if needed)
df['pain'] = df['pain'].round()

# Convert the 'pain' column to integers
df['pain'] = df['pain'].astype(int)

In [0]:
df['pain'].value_counts()

###acuity

In [0]:
df['acuity'].value_counts()

We obtained the expected distribution of data based on urgency levels. The majority of patients are categorized at level 3, while the fewest are categorized at level 5.

For this reason, we will need to later augment the data to achieve balance across the urgency levels.

Visualization of urgency levels.

In [0]:
plt.figure(figsize=(10, 6))
sns.countplot(x='acuity', data=df, palette='viridis')
plt.title('Number of Patients per Urgency Level')
plt.xlabel('Urgency Level')
plt.ylabel('Number of Patients')
plt.grid(True)
plt.savefig('number_of_patients_per_urgency_level.png')
plt.show()

###chiefcomplaint

In [0]:
df['chiefcomplaint'].value_counts().head(20)

In [0]:
most_common_complaints = {}

# List of acuity levels (you can adjust this based on your actual acuity levels)
acuity_levels = df['acuity'].unique()

# Iterate over each acuity level
for level in acuity_levels:
    # Filter the DataFrame for the current acuity level
    df_level = df[df['acuity'] == level]

    # Count the frequency of each chief complaint
    complaint_counts = df_level['chiefcomplaint'].value_counts()

    # Store the result in the dictionary
    most_common_complaints[level] = complaint_counts

# Display the results
for level, counts in most_common_complaints.items():
    print(f"Most common chief complaints for acuity level {level}:")
    print(counts)
    print()

In [0]:
# Count the number of unique values in the 'chiefcomplaint' column
unique_values_count = df['chiefcomplaint'].nunique()

# Print the number of unique values
print(f'Number of unique values in the "chiefcomplaint" column: {unique_values_count}')

In [0]:
# Convert the 'chiefcomplaint' column to lowercase
df['chiefcomplaint'] = df['chiefcomplaint'].str.lower()

# Split symptoms using comma as the delimiter
# Create a list of all symptoms
symptoms = df['chiefcomplaint'].str.split(',\s*').explode()

# Calculate the frequency of each symptom
symptom_counts = symptoms.value_counts()

# Get the top 60 most common symptoms
top_60_symptoms = symptom_counts.head(60)

# Print the top 60 most common symptoms
print('Top 60 most common symptoms:')
print(top_60_symptoms)

In [0]:
# Define a dictionary to store symptom counts by acuity level
symptom_counts_by_acuity = {}

# Get unique acuity levels
acuity_levels = df['acuity'].unique()

# Analyze symptoms for each acuity level
for level in acuity_levels:
    # Filter the DataFrame for the current acuity level
    df_level = df[df['acuity'] == level]

    # Split symptoms using comma as the delimiter and create a list of symptoms
    symptoms = df_level['chiefcomplaint'].str.split(',\s*').explode()

    # Calculate the frequency of each symptom
    symptom_counts = symptoms.value_counts()

    # Store the result in the dictionary
    symptom_counts_by_acuity[level] = symptom_counts

# Display the results for each acuity level
for level, counts in symptom_counts_by_acuity.items():
    print(f'Top symptoms for acuity level {level}:')
    print(counts.head(10))  # Adjust the number if you need more or fewer top symptoms
    print()

In [0]:
# Define a dictionary to store symptom counts by acuity level
symptom_counts_by_acuity = {}

# Define a counter to count how many symptoms have more than 10 occurrences
symptoms_over_10_count = 0

# Get unique acuity levels
acuity_levels = df['acuity'].unique()

# Analyze symptoms for each acuity level
for level in acuity_levels:
    # Filter the DataFrame for the current acuity level
    df_level = df[df['acuity'] == level]

    # Split symptoms using comma as the delimiter and create a list of symptoms
    symptoms = df_level['chiefcomplaint'].str.split(',\s*').explode()

    # Calculate the frequency of each symptom
    symptom_counts = symptoms.value_counts()

    # Increment the counter for symptoms with more than 10 occurrences
    symptoms_over_10_count += (symptom_counts > 10).sum()

    # Store the result in the dictionary
    symptom_counts_by_acuity[level] = symptom_counts

# Display the results for each acuity level
for level, counts in symptom_counts_by_acuity.items():
    print(f'Top symptoms for acuity level {level}:')
    print(counts.head(10))  # Adjust the number if you need more or fewer top symptoms
    print()

# Print the total count of symptoms with more than 10 occurrences
print(f'Total number of symptoms with more than 10 occurrences: {symptoms_over_10_count}')

we will use this later for 'Data manipulation':

In [0]:
import pandas as pd

# Assuming the DataFrame is already loaded into df

# Define conditions for non-critical parameters
non_critical_conditions = (
    ~((df['temperature_celsius'] < 32) | (df['temperature_celsius'] > 40)) &  # Temperature conditions
    ~((df['heartrate'] < 40) | (df['heartrate'] > 150)) &  # Heartrate conditions
    ~((df['resprate'] < 8) | (df['resprate'] > 30)) &  # Respiratory rate conditions
    ~((df['o2sat'] < 90)) &  # O2 saturation conditions
    ~((df['sbp'] < 90)) &  # SBP conditions
    ~(df['pain'] == 10)  # Pain condition
)

# Define conditions for rows where the acuity level is 1
acuity_1 = df['acuity'] == 1

# Combine both sets of conditions: acuity level 1 and non-critical parameters
valid_rows = acuity_1 & non_critical_conditions

# Filter the DataFrame based on the conditions
df_valid = df[valid_rows]

# Print the values in the 'chiefcomplaint' column for the filtered rows
critical_chiefcomplaint = df_valid['chiefcomplaint'].unique()

In [0]:
critical_chiefcomplaint.shape

In [0]:
chiefcomplaints_class_5 = df.loc[df['acuity'] == 5, 'chiefcomplaint'].unique()

In [0]:
chiefcomplaints_class_5.shape

One-hot encoding.(TF-IDF produces same results)

In [0]:
from collections import Counter


acuity_sizes = df['acuity'].value_counts()


percentage_to_keep = 0.01


thresholds = {}
for level in acuity_sizes.index:
    size = acuity_sizes[level]
    threshold = max(1, int(size * percentage_to_keep))
    thresholds[level] = threshold

symptoms_per_level = {}

for level in acuity_sizes.index:

    df_level = df[df['acuity'] == level]

    all_words = df_level['chiefcomplaint'].str.lower().str.split(',\s*').explode()


    word_counts = Counter(all_words)


    significant_words = [word for word, count in word_counts.items() if count >= thresholds[level]]


    symptoms_per_level[level] = significant_words

all_significant_words = set(word for words in symptoms_per_level.values() for word in words)


symptoms_count_per_level = {level: len(symptoms_per_level.get(level, [])) for level in acuity_sizes.index}


for word in all_significant_words:

    df[word] = df['chiefcomplaint'].str.lower().apply(lambda x: 1 if word in x.split(', ') else 0)

for level, count in symptoms_count_per_level.items():
    print(f'Acuteness levels {level} will have {count} new columns.')



df.head()

In [0]:
df = df.drop(columns=['chiefcomplaint'])

###gender

In [0]:
df['gender'].value_counts()

In [0]:
df['gender'] = df['gender'].map({'F': 0, 'M': 1})

###arrival transport

In [0]:
df['arrival_transport'].value_counts()

In [0]:
df['arrival_transport'] = df['arrival_transport'].map({
    'WALK IN': 0,
    'AMBULANCE': 3,
    'UNKNOWN': 1,
    'OTHER': 2,
    'HELICOPTER': 4
})

In [0]:
df['arrival_transport'].value_counts()

###race

In [0]:
df = pd.get_dummies(df, columns=['race'], drop_first=True)

###sbp and dbp

In [0]:
df= df[df['dbp'] < df['sbp']]

In [0]:
# Assuming 'df' is a DataFrame and it contains a column 'sbp'
def map_sbp(sbp):
    # Map systolic blood pressure (SBP) values to categories:
    # If SBP is less than 90, classify as 0 (Abnormal)
    if sbp < 90:
        return 0  # Abnormal
    # If SBP is between 90 and 119 (inclusive), classify as 1 (Normal)
    elif sbp < 120:
        return 1  # Normal
    # If SBP is between 120 and 129 (inclusive), classify as 2 (Elevated)
    elif 120 <= sbp <= 129:
        return 2  # Elevated
    # If SBP is between 130 and 139 (inclusive), classify as 3 (Hypertension Stage 1)
    elif 130 <= sbp <= 139:
        return 3  # Hypertension Stage 1
    # If SBP is between 140 and 179 (inclusive), classify as 4 (Hypertension Stage 2)
    elif 140 <= sbp < 180:
        return 4  # Hypertension Stage 2
    # If SBP is 180 or higher, classify as 5 (Hypertensive Crisis)
    else:
        return 5  # Hypertensive Crisis

# Add a new column with categories based on SBP values
df['sbp_class'] = df['sbp'].apply(map_sbp)

In [0]:
# Assuming 'df' is a DataFrame and it contains a column 'dbp'
def map_dbp(dbp):
    # Map diastolic blood pressure (DBP) values to categories:
    # If DBP is less than 60, classify as 0 (Abnormal)
    if dbp < 60:
        return 0  # Abnormal
    # If DBP is between 60 and 79 (inclusive), classify as 1 (Normal)
    elif dbp < 80:
        return 1  # Normal
    # If DBP is between 80 and 89 (inclusive), classify as 2 (Hypertension Stage 1)
    elif 80 <= dbp <= 89:
        return 2  # Hypertension Stage 1
    # If DBP is between 90 and 119 (inclusive), classify as 3 (Hypertension Stage 2)
    elif 90 <= dbp < 120:
        return 3  # Hypertension Stage 2
    # If DBP is 120 or higher, classify as 4 (Hypertensive Crisis)
    else:
        return 4  # Hypertensive Crisis

# Add a new column with categories based on DBP values
df['dbp_class'] = df['dbp'].apply(map_dbp)

###heartrate

In [0]:
# Count the number of rows before filtering
rows_before = df.shape[0]

# Apply the filtering condition
df = df[~((df['heartrate'] > 400) | (df['heartrate'] < 30))]

# Count the number of rows after filtering
rows_after = df.shape[0]

# Calculate the number of rows removed
rows_removed = rows_before - rows_after

# Print the result
print(f"Number of rows removed: {rows_removed}")

In [0]:
def map_heartrate(rate):
    # Map heart rate values to categories:
    # If the rate is less than 60, classify as 0 (bradycardia)
    if rate < 60:
        return 0  # Bradycardia
    # If the rate is between 60 and 100 (inclusive), classify as 1 (normal)
    elif 60 <= rate <= 100:
        return 1  # Normal
    # If the rate is greater than 100, classify as 2 (tachycardia)
    else:
        return 2  # Tachycardia

# Add a new column with categories based on the heart rate values
df['heartrate_map'] = df['heartrate'].apply(map_heartrate)

##Data manipulation-removing additonal outliers

We are excluding outliers, meaning patients with critical measurements who are not classified as urgency level 1 or at least level 2.

In [0]:
df.shape

In [0]:
# Define conditions for critical parameters
critical_conditions = (
    (df['temperature_celsius'] < 32) | (df['temperature_celsius'] > 40) |  # Temperature conditions
    (df['heartrate'] < 40) | (df['heartrate'] > 150) |  # Heartrate conditions
    (df['resprate'] < 8) | (df['resprate'] > 30) |  # Resprate conditions
    (df['o2sat'] < 90) |  # O2sat condition
    (df['sbp'] < 90) |  # SBP condition
    (df['pain'] >= 10)  # Pain condition
)

# Define conditions for rows where acuity level is 1 or 2
acuity_1_2 = df['acuity'].isin([1, 2])

# Combine conditions: We want to drop rows where acuity is not 1 or 2 and have critical parameters
rows_to_drop = ~acuity_1_2 & critical_conditions

# Filter the DataFrame - keep all rows that do not meet the drop condition
df = df[~rows_to_drop]

In [0]:
df.shape

Remove rows from df where at least one column listed in 'critical_chiefcomplaint' is equal to 1,and the row does not have a column 'acuity' equal to 1

Note: 'critical_chiefcomplaint' was previously defined as the chief complaint from acuity level 1,
where all other parameters were within the normal range, leading us to conclude that these chief complaints
are critical on their own.

In [0]:
# Check which columns from critical_chiefcomplaint exist in the DataFrame
existing_columns = [col for col in critical_chiefcomplaint if col in df.columns]

In [0]:
len(existing_columns)

In [0]:
existing_columns

In [0]:
# Create a condition for the rows to exclude
rows_to_exclude = df[
    df[existing_columns].apply(lambda row: any(row == 1), axis=1) & (df['acuity'] != 1)
]

# Exclude those rows from the DataFrame
df = df[~df.index.isin(rows_to_exclude.index)]

In [0]:
df.shape

We are removing from all other classes those records that have a note from acuity class 5, where this note from acuity class 5 has somewhat critical parameters (a bit broader than the previously defined critical parameters), but these records are still categorized as class 5, indicating that they are not as critical.

In [0]:
chiefcomplaints_class_5 = [col for col in chiefcomplaints_class_5 if col in df.columns]

In [0]:
print(df.columns.tolist())

In [0]:
chiefcomplaint=['resprate1', 'heartrate1', 'weakness', 'bradycardia', 'diarrhea', 'staples removal',
                'rash', 'slurred speech', 'ili', 'hyperglycemia', 'sore throat', 'si', 'transfer',
                'syncope', 'headache', 'depression', 'r ankle pain', 'dental pain', 'r shoulder pain',
                'confusion', 'r foot pain', 'l ankle pain', 'rabies vaccine', 'etoh', 'cough',
                'r knee pain', 'brbpr', 'neck pain', 'abd pain', 'mvc', 'l knee pain', 'palpitations',
                'dysuria', 'tachycardia', 'abnormal labs', 'wound eval', 'rlq abdominal pain',
                'l foot pain', 'dyspnea', 'nausea', 'suture removal', 'altered mental status', 'n/v',
                'seizure', 'hypotension', 'visual changes', 'finger laceration', 'l ear pain',
                'laceration', 'hypoxia', 'allergic reaction', 'vomiting', 'fever', 's/p fall',
                'toe pain', 'med refill', 'chest pain', 'back pain', 'lower back pain', '___',
                'n/v/d', 'dizziness']

In [0]:
chiefcomplaint_without_chiefcomplaints_class_5 = list(set(chiefcomplaint) - set(chiefcomplaints_class_5))

In [0]:
# Define conditions for critical parameters
critical_conditions = (
    (df['temperature_celsius'] < 34) | (df['temperature_celsius'] > 38) |  # Temperature conditions
    (df['heartrate'] < 50) | (df['heartrate'] > 140) |  # Heartrate conditions
    (df['resprate'] < 8) | (df['resprate'] > 30) |  # Resprate conditions
    (df['o2sat'] < 92) |  # O2sat condition
    (df['sbp'] < 92) |  # SBP condition
    (df['pain'] > 8)  # Pain condition
)

# Identify all rows in acuity levels 1, 2, 3, 4 with at least one column in chiefcomplaints_class_5 equal to 1,
# that have critical conditions, and none of the columns in chiefcomplaint_without_chiefcomplaints_class_5 are equal to 1
rows_to_drop = (
    df['acuity'].isin([1, 2, 3, 4]) &  # Condition for acuity being in levels 1, 2, 3, 4
    df[chiefcomplaints_class_5].eq(1).any(axis=1) &  # Condition for at least one column in chiefcomplaints_class_5 being equal to 1
    critical_conditions &  # Critical conditions
    ~df[chiefcomplaint_without_chiefcomplaints_class_5].eq(1).any(axis=1)  # Condition for none of the columns in chiefcomplaint_without_chiefcomplaints_class_5 being equal to 1
)

# Filter the DataFrame - keep all rows that do not meet the exclusion condition
df = df[~rows_to_drop]

In [0]:
df.shape

In [0]:
df = df[df['___'] != 1]
df.drop('___', axis=1, inplace=True)

##Feature engineering

In [0]:
df['pulse_pressure'] = df['sbp'] - df['dbp']  # Difference between systolic and diastolic blood pressure
df['oxygen_ratio'] = df['o2sat'] / df['resprate']  # Ratio of oxygen saturation to respiratory rate
# Examples of interaction features
df['temp_pulse_pressure'] = df['temperature_celsius'] * df['pulse_pressure']  # Interaction between temperature and pulse pressure

In [0]:
# Check and handle NaN and inf values
df.replace([np.inf, -np.inf], np.nan, inplace=True)  # Replace inf with NaN
df.dropna(inplace=True)  # Remove rows with NaN

##Model

###Data scaling and splitting into train and test sets

In [0]:
# Separate the columns that you do not want to scale
non_scaling_columns = ['subject_id_x', 'acuity']
scaling_columns = [col for col in df.columns if col not in non_scaling_columns]

# Extract the portion of the DataFrame that you want to scale
X = df[scaling_columns]

from sklearn.preprocessing import StandardScaler
# Create an instance of StandardScaler and apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create a new DataFrame with the scaled data
df_scaled = pd.DataFrame(X_scaled, columns=scaling_columns)

# Reattach the original columns
df_scaled[non_scaling_columns] = df[non_scaling_columns].values

# Reorganize the columns if necessary
df_scaled = df_scaled[df.columns]

In [0]:
X = df_scaled.drop(['acuity'], axis=1)  # Features
y = df_scaled['acuity']  # Target variable

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

###Random forest

In [0]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [0]:
class_weights = {1: 2, 2: 1, 3: 1, 4: 2, 5: 2}

from sklearn.model_selection import KFold, cross_val_score

model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight=class_weights, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='precision_weighted')

# Print results
print(f"Precision scores for each fold: {cv_scores}")
print(f"Average precision: {np.mean(cv_scores)}")

In [0]:
model.fit(X_train, y_train)

In [0]:
y_pred = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)

In [0]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test, y_pred, labels=[1], average='macro')

# Print results
print(f'Overall Accuracy: {accuracy:.2f}')
print(f'Precision for class 1: {precision_class_1:.2f}')
print('Classification Report:')
print(report)

In [0]:

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision_class_1 = precision_score(y_test, y_pred, labels=[1], average='macro')
report = classification_report(y_test, y_pred)

# Set the style for the plot
sns.set(style="whitegrid")

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Define green shades
light_green = '#98fb98'      # PaleGreen
medium_green = '#90ee90'     # LightGreen
lightest_green = '#f0fff0'   # Honeydew

# Display results in colored boxes with green shades
ax.text(0.05, 0.75, f'Overall Accuracy: {accuracy:.2f}', fontsize=12, bbox=dict(facecolor=light_green, alpha=0.6))
ax.text(0.05, 0.55, f'Precision for class 1: {precision_class_1:.2f}', fontsize=12, bbox=dict(facecolor=light_green, alpha=0.6))
ax.text(0.05, 0.1, f'Classification Report:\n{report}', fontsize=10, bbox=dict(facecolor=light_green, alpha=0.6), ha='left', va='top')

# Remove axes and add title
ax.axis('off')
ax.set_title("Model Evaluation Results", fontsize=16, pad=20)

# Show the plot
plt.tight_layout()
plt.show()


In [0]:
cm = confusion_matrix(y_test, y_pred, labels=[1, 2, 3, 4, 5])

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[1, 2, 3, 4, 5], yticklabels=[1, 2, 3, 4, 5])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [0]:
y_test_combined = y_test.copy()
y_pred_combined = y_pred.copy()

# Change values 4 and 5 to 3 using np.isin
y_test_combined[np.isin(y_test_combined, [4, 5])] = 3
y_pred_combined[np.isin(y_pred_combined, [4, 5])] = 3

# Re-evaluate after combining classes
accuracy = accuracy_score(y_test_combined, y_pred_combined)
report = classification_report(y_test_combined, y_pred_combined)

# Calculate precision specifically for class 1
precision_class_1 = precision_score(y_test_combined, y_pred_combined, labels=[1], average='macro')

# Print results with emphasis on precision for class 1
print(f'Overall Accuracy: {accuracy:.2f}')
print(f'Precision for class 1: {precision_class_1:.2f}')
print('Classification Report:')
print(report)

In [0]:
# Define the mapping function
def map_acuity(value):
    if value in [1]:
        return 'Urgent'
    else:
        return 'Non urgent'

# Apply the mapping function to actual and predicted values
y_test_mapped = pd.Series(y_test).apply(map_acuity)
y_pred_mapped = pd.Series(y_pred).apply(map_acuity)

# Calculate accuracy and classification report for the mapped classes
accuracy_mapped = accuracy_score(y_test_mapped, y_pred_mapped)
report_mapped = classification_report(y_test_mapped, y_pred_mapped, target_names=['Non urgent', 'Urgent'])

# Calculate precision specifically for the 'Hitno' class
precision_urgent = precision_score(y_test_mapped, y_pred_mapped, pos_label='Urgent')

# Print results with emphasis on precision for the 'Hitno' class
print(f'Accuracy for Mapped Classes: {accuracy_mapped:.2f}')
print(f'Precision for class "Urgent": {precision_urgent:.2f}')
print('Classification Report for Mapped Classes:')
print(report_mapped)

In [0]:
report = classification_report(y_test_mapped, y_pred_mapped, target_names=['Non urgent', 'Urgent'], output_dict=True)

metrics = ['precision', 'recall', 'f1-score']
classes = ['Non urgent', 'Urgent']
data = {metric: [report[cls][metric] for cls in classes] for metric in metrics}

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(classes))
width = 0.2

for i, metric in enumerate(metrics):
    ax.bar([p + width*i for p in x], data[metric], width=width, label=metric)

ax.set_xlabel('Classes')
ax.set_ylabel('Scores')
ax.set_title('Classification Metrics')
ax.set_xticks([p + width for p in x])
ax.set_xticklabels(classes)
ax.legend()

plt.show()

In [0]:
cm = confusion_matrix(y_test_mapped, y_pred_mapped, labels=['Non urgent', 'Urgent'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non urgent', 'Urgent'], yticklabels=['Non urgent', 'Urgent'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [0]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Assuming y_test_mapped and y_pred_mapped are defined and mapped correctly
precision, recall, _ = precision_recall_curve(
    y_test_mapped.map({'Urgent': 1, 'Non urgent': 0}),
    y_pred_mapped.map({'Urgent': 1, 'Non urgent': 0})
)

# Plot Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True)
plt.show()


In [0]:
from sklearn.metrics import roc_curve, auc

# Convert labels to binary form
y_test_binary = y_test_mapped.apply(lambda x: 1 if x == 'Urgent' else 0)
y_pred_binary = y_pred_mapped.apply(lambda x: 1 if x == 'Urgent' else 0)

# Assuming you have probabilities for the positive class
# If not, use the model to predict probabilities
# y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probabilities for the positive class

# Using binary values as an example
# If you do not have probabilities, replace with your model's probabilities
y_pred_proba = pd.Series([0.8 if x == 'Urgent' else 0.2 for x in y_pred_mapped])

# Calculate the ROC curve
fpr, tpr, thresholds = roc_curve(y_test_binary, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.show()

# Print results with description and comments based on AUC
print(f'Area Under the Curve (AUC): {roc_auc:.2f}')

# Comment based on AUC value
if roc_auc >= 0.9:
    print('Excellent model performance: The ROC AUC score is very high, indicating that the model is excellent at distinguishing between the "Urgent" class and other classes.')
elif roc_auc >= 0.8:
    print('Good model performance: The ROC AUC score is high, suggesting that the model performs well in distinguishing between the "Urgent" class and other classes.')
elif roc_auc >= 0.7:
    print('Moderate model performance: The ROC AUC score indicates that the model has a fair ability to distinguish between the "Urgent" class and other classes.')
else:
    print('Poor model performance: The ROC AUC score is low, which suggests that the model has limited ability to distinguish between the "Urgent" class and other classes.')

print(f'The ROC curve shows the trade-off between the True Positive Rate (TPR) and False Positive Rate (FPR).')
print(f'An AUC value of {roc_auc:.2f} indicates the model\'s ability to distinguish between the "Urgent" class and other classes.')
print(f'A value of 1.0 represents a perfect model, while a value of 0.5 represents a model with no discrimination ability (random guessing).')

#Fourth method for addressing the problem

In [0]:
from google.colab import drive
drive.mount('/content/drive')

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import pickle

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

%cd /content/drive/MyDrive/data
df1 = pd.read_csv('triage.csv')
df2=pd.read_csv('edstays.csv')
df = pd.merge(df1, df2, on='stay_id', how='left')

In [0]:
df.drop(columns=['gender','race','subject_id_x','stay_id','subject_id_y','hadm_id','intime','outtime','disposition','pain'],inplace=True)

In [0]:

# Assuming df is already loaded and processed as described earlier

# Drop rows where 'chiefcomplaint' is NaN
df = df.dropna(subset=['chiefcomplaint'])

# Filter rows where 'chiefcomplaint' starts with a letter or quotation mark (")
df = df[df['chiefcomplaint'].str.startswith(tuple('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"'))]

# Remove the leading quotation mark from 'chiefcomplaint' entries starting with it
df['chiefcomplaint'] = df['chiefcomplaint'].apply(lambda x: x[1:] if x.startswith('"') else x)

# Extract text before the first comma in 'chiefcomplaint'
df['chiefcomplaint'] = df['chiefcomplaint'].apply(lambda x: x.split(',')[0])

# Normalize text by stripping spaces and converting to uppercase
df['chiefcomplaint'] = df['chiefcomplaint'].str.strip().str.upper()

# Remove duplicates, keeping only the first occurrence
df = df.drop_duplicates(subset=['chiefcomplaint'], keep='first')

# Display the resulting DataFrame
print("DataFrame after removing duplicates and keeping only the first occurrence:")
print(df)

# Count the number of duplicates for each unique 'chiefcomplaint'
duplicate_counts = df['chiefcomplaint'].value_counts()
print("\nCount of each unique 'chiefcomplaint' value (duplicates only):")
print(duplicate_counts[duplicate_counts > 1])

In [0]:
# Remove duplicates, keeping only the first occurrence
df = df.drop_duplicates(subset=['chiefcomplaint'], keep='first')

# Display the resulting DataFrame
print("DataFrame after removing duplicates and keeping only the first occurrence:")
print(df)

In [0]:
from sklearn.preprocessing import LabelEncoder

# Define the LabelEncoders
label_encoders = {
    'chiefcomplaint': LabelEncoder(),
    'arrival_transport': LabelEncoder(),
}

# Assuming df is your DataFrame
# Apply LabelEncoder to each categorical column and extract the mappings
mappings = {}

for col, le in label_encoders.items():
    df[col] = le.fit_transform(df[col])
    mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

# Display the encoded data
print("Encoded DataFrame:")
print(df)

# Display the mapping for chiefcomplaint
print("\nChief Complaint Mapping:")
print(mappings['chiefcomplaint'])

import os

# Get the current working directory
current_directory = os.getcwd()

# Create the file path in the current working directory
file_path = os.path.join(current_directory, 'chiefcomplaint_mapping_latest1.txt')

# Save the 'chiefcomplaint' mapping to a text file in the current working directory
with open(file_path, 'w') as f:
    for complaint, code in mappings['chiefcomplaint'].items():
        f.write(f'{code}: {complaint}\n')

print(f"File saved to {file_path}")


print("Chief complaint mapping has been saved to 'chiefcomplaint_mapping.txt'.")

In [0]:
data=df.dropna()

In [0]:
# Convert the 'acuity' column from float to integer
data['acuity'] = data['acuity'].astype(int)

# Get the unique values in the 'acuity' column
unique_acuity_values = data['acuity'].unique()

# Sort the unique values for better readability
unique_acuity_values.sort()

# Display the unique values
print("Unique values in the 'acuity' column:")
print(unique_acuity_values)

In [0]:
# Convert the temperature from Fahrenheit to Celsius
data['temperature'] = (data['temperature'] - 32) * 5.0/9.0
# Round the temperature column to one decimal place
data['temperature'] = data['temperature'].round(1)
# Display the first few rows to confirm the transformation
data.head(5)

In [0]:
# Save the encoders for later use in production
with open('label_encoders_latest.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

In [0]:
data.head()

In [0]:
# Features and target variable
X = data.drop(columns=['acuity'])
y = data['acuity']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)

In [0]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:\n", report)

In [0]:
# Acuity 1: Emergency cases, likely higher severity
patient_1 = {
    'temperature': 39.0, 'heartrate': 120, 'resprate': 30, 'o2sat': 92,
    'sbp': 85, 'dbp': 55, 'chiefcomplaint': 1360,  # 'ABD PAIN AND VOMTIING'
    'arrival_transport': 0  # Female, Ambulance
}

patient_2 = {
    'temperature': 38.5, 'heartrate': 110, 'resprate': 28, 'o2sat': 94,
    'sbp': 90, 'dbp': 60, 'chiefcomplaint': 1356,  # 'ABD PAIN AND PREGNANT'
     'arrival_transport': 0  # Female, Ambulance
}

# Acuity 2: Urgent cases, still critical but slightly less severe
patient_3 = {
    'temperature': 37.8, 'heartrate': 105, 'resprate': 24, 'o2sat': 95,
    'sbp': 100, 'dbp': 65, 'chiefcomplaint': 1240,  # 'ABD CRAMPS AND FEVER'
     'arrival_transport': 0  # Female, Ambulance
}

patient_4 = {
    'temperature': 38.0, 'heartrate': 102, 'resprate': 22, 'o2sat': 96,
    'sbp': 95, 'dbp': 60, 'chiefcomplaint': 1193,  # 'ABD PAIN /HYPOTENSION'
     'arrival_transport': 1  # Male, Walk-in
}

# Acuity 3: Moderate cases, requiring prompt but not immediate attention
patient_5 = {
    'temperature': 37.5, 'heartrate': 95, 'resprate': 20, 'o2sat': 97,
    'sbp': 110, 'dbp': 70, 'chiefcomplaint': 1189,  # 'ABD PAIN  DIARRHEA'
     'arrival_transport': 0  # Male, Ambulance
}

patient_6 = {
    'temperature': 37.2, 'heartrate': 88, 'resprate': 18, 'o2sat': 98,
    'sbp': 115, 'dbp': 75, 'chiefcomplaint': 1287,  # 'ABD PAIN'
     'arrival_transport': 1  # Female, Walk-in
}

# Acuity 4: Less severe cases, could likely wait a bit longer for treatment
patient_7 = {
    'temperature': 37.0, 'heartrate': 85, 'resprate': 16, 'o2sat': 99,
    'sbp': 120, 'dbp': 80, 'chiefcomplaint': 1275,  # 'ABD HERNIA/TRANSFER'
     'arrival_transport': 1  # Male, Walk-in
}

patient_8 = {
    'temperature': 36.9, 'heartrate': 80, 'resprate': 15, 'o2sat': 100,
    'sbp': 125, 'dbp': 85, 'chiefcomplaint': 1457,  # 'ABD PAIN, ERCP'
     'arrival_transport': 1  # Female, Walk-in
}

# Acuity 5: Non-urgent cases, lowest priority for treatment
patient_9 = {
    'temperature': 36.6, 'heartrate': 75, 'resprate': 14, 'o2sat': 100,
    'sbp': 130, 'dbp': 90, 'chiefcomplaint': 1150,  # 'ABCESS ON FACE'
     'arrival_transport': 1  # Male, Walk-in
}

patient_10 = {
    'temperature': 36.7, 'heartrate': 72, 'resprate': 14, 'o2sat': 100,
    'sbp': 128, 'dbp': 88, 'chiefcomplaint': 1153,  # 'ABCESS R ARM'
     'arrival_transport': 1  # Female, Walk-in
}

In [0]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Assuming you already have the model trained as in your example:
# (if not, you would need to train the model first using the code you provided above)

# Convert the patient data into a DataFrame
patients = pd.DataFrame([
    {
        'temperature': 39.0, 'heartrate': 120, 'resprate': 30, 'o2sat': 92,
        'sbp': 85, 'dbp': 55, 'chiefcomplaint': 1360,
         'arrival_transport': 0
    },
    {
        'temperature': 38.5, 'heartrate': 110, 'resprate': 28, 'o2sat': 94,
        'sbp': 90, 'dbp': 60, 'chiefcomplaint': 1356,
         'arrival_transport': 0
    },
    {
        'temperature': 37.8, 'heartrate': 105, 'resprate': 24, 'o2sat': 95,
        'sbp': 100, 'dbp': 65, 'chiefcomplaint': 1240,
         'arrival_transport': 0
    },
    {
        'temperature': 38.0, 'heartrate': 102, 'resprate': 22, 'o2sat': 96,
        'sbp': 95, 'dbp': 60, 'chiefcomplaint': 1193,
         'arrival_transport': 1
    },
    {
        'temperature': 37.5, 'heartrate': 95, 'resprate': 20, 'o2sat': 97,
        'sbp': 110, 'dbp': 70, 'chiefcomplaint': 1189,
         'arrival_transport': 0
    },
    {
        'temperature': 37.2, 'heartrate': 88, 'resprate': 18, 'o2sat': 98,
        'sbp': 115, 'dbp': 75, 'chiefcomplaint': 1287,
         'arrival_transport': 1
    },
    {
        'temperature': 37.0, 'heartrate': 85, 'resprate': 16, 'o2sat': 99,
        'sbp': 120, 'dbp': 80, 'chiefcomplaint': 1275,
         'arrival_transport': 1
    },
    {
        'temperature': 36.9, 'heartrate': 80, 'resprate': 15, 'o2sat': 100,
        'sbp': 125, 'dbp': 85, 'chiefcomplaint': 1457,
         'arrival_transport': 1
    },
    {
        'temperature': 36.6, 'heartrate': 75, 'resprate': 14, 'o2sat': 100,
        'sbp': 130, 'dbp': 90, 'chiefcomplaint': 1150,
         'arrival_transport': 1
    },
    {
        'temperature': 36.7, 'heartrate': 72, 'resprate': 14, 'o2sat': 100,
        'sbp': 128, 'dbp': 88, 'chiefcomplaint': 1153,
         'arrival_transport': 1
    }
])

# Make predictions
predictions = model.predict(patients)

# Print the predictions
for i, prediction in enumerate(predictions, 1):
    print(f"Patient {i} predicted acuity level: {prediction}")


In [0]:
from joblib import dump, load
dump(model, 'model_latest1.joblib')
